# SUMMER camera daemon interface test

Mirror of `spring_camera_daemon_interface_test.ipynb` for the SUMMER camera.

Prerequisites (on the summer camera computer):
1. The GUI server is running:
   `python -m cmos_camera_gui --headless --ws-port 5566 --auto-connect qhy --instrument summer`
2. The summer camera daemon is running:
   `python wsp/camera/daemons/summer_camera_daemon.py -n <ns_host>`
3. A Pyro5 nameserver is reachable (local `-n localhost` for bench testing).

In [ ]:
from wsp.utils.paths import WSP_PATH, CONFIG_PATH
from wsp.utils.utils import loadconfig
from datetime import datetime
from wsp.camera.implementations.summer_camera import SummerCamera

config = loadconfig(CONFIG_PATH)

cam = SummerCamera(
    base_directory=WSP_PATH,
    config=config,
    camname="summer",
    daemon_pyro_name="SUMMERCamera",
    ns_host_camera="localhost",
    ns_host_hk="192.168.1.10",
    logger=None,
    verbose=False,
)

## Robotic startup: set TEC setpoint from config and cool until stable

Should go `OFF -> STARTUP_REQUESTED`, then `-> READY` once
`|tec_temp - setpoint| < 0.5` (measured ~2 min from warm to a locked 0 C).

In [ ]:
cam.startupCamera()

In [ ]:
cam.update_state()
cam.state["camera_state"]

## doExposure round trip

Verifies the full chain: daemon `set_save_path` + `capture_frames` ->
GUI writes `<imdir>/<imname>.fits` -> daemon sees `is_capturing` drop,
symlinks the last image, and returns to READY.

In [ ]:
import time

datestr = datetime.now().strftime("%Y%m%d")
imdir = f"~/data/images/{datestr}/summer"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
imname = f"test_{timestamp}"

print(f"Camera state:")
print(f"Attempting to take image {imname} in {imdir}")
cam.update_state()
print(f"Camera State: {cam.state['camera_state']}")
sendtime = time.time()
cam.doExposure(imdir=imdir, imname=imname)
print(f"Camera State: {cam.state['camera_state']}")
print("Waiting for camera to be ready...")
while cam.state["camera_state"] != "READY":
    cam.update_state()
endtime = time.time()

print(f"Camera State: {cam.state['camera_state']}")
print(f"Camera exposure took {endtime - sendtime:.1f} seconds")

## setExposure round trip

Completion requires `gui_state == 'READY'` AND the exact float echo of the
requested exposure time in status (`exptime`).

In [ ]:
import time

exptime = 7.5  # seconds
print(f"Camera state:")
print(f"Attempting to set exposure time to {exptime} s")
cam.update_state()
print(f"Camera State: {cam.state['camera_state']}")
sendtime = time.time()
cam.setExposure(exptime)
print(f"Camera State: {cam.state['camera_state']}")
print("Waiting for camera to be ready...")
while cam.state["camera_state"] != "READY":
    cam.update_state()
endtime = time.time()

print(f"Camera State: {cam.state['camera_state']}")
print(f"Setting exposure time took {endtime - sendtime:.1f} seconds")

In [ ]:
cam.update_state()
print(f"GUI State: {cam.state.get('gui_state', 'UNKNOWN')}")

In [ ]:
cam.print_state()

## TEC management through the daemon

In [ ]:
# set a new setpoint (clamped to -45..+20 C) and watch it settle
cam.tecSetSetpoint(-5.0)

In [ ]:
cam.update_state()
{k: cam.state.get(k) for k in (
    "tec_temp", "tec_setpoint", "tec_enabled", "tec_steady",
    "tec_power_pct", "camera_state", "gui_state",
)}

## Direct GUI check (bypasses the daemon)

Ground-truth `get_status` reply straight from the GUI server -- compare to
WspSummerDaemonHandoff.md section 4.

In [ ]:
from cmos_camera_gui.summer_client import SummerClient

cc = SummerClient("localhost", 5566)
cc.connect()

cc.get_status()

In [ ]:
cc.set_exposure(10)

In [ ]:
cc.get_status()

## Robotic shutdown

Completes as soon as the TEC reports disabled (no warm-up wait on SUMMER).

In [ ]:
cam.shutdownCamera()